# H&M 2년 M4 음성 포화 사전 점검
같은 점검을 H&M 2년 M1 체크포인트에 적용합니다. 두 데이터 모두에서 조건을 충족할 때만 CLV 조건부 음성추출 M4를 설계합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = 'f9fd2d075e5b2dd19f2f460f21239d004f22f49f'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import importlib
import json
import torch
import lightgcn_clv_m4_negative_saturation_diagnostic as saturation
saturation = importlib.reload(saturation)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert saturation.CODE_VERSION == 'm4-negative-saturation-diagnostic-v1'
cfg = saturation.configure_negative_saturation_diagnostic('hm')
print(json.dumps(saturation.preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
paths = saturation.run_negative_saturation_diagnostic(
    saturation.configure_negative_saturation_diagnostic('hm')
)


In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(paths['summary_csv'])
print('1) CLV 구간별 학습신호')
display(summary)
report = json.load(open(paths['json']))
print('2) 고CLV - 저CLV 95% bootstrap 구간')
display(pd.DataFrame(report['bootstrap']['contrasts']).T)
print('3) 판독')
print(json.dumps(report['reading'], ensure_ascii=False, indent=2))
print('판독 기준: 고CLV의 균등 음성 학습신호가 저CLV보다 낮고, 어려운 음성으로 바꿨을 때의 증가분은 고CLV가 더 커야 함 (두 데이터 모두, 구간이 0을 제외)')
print('결과 파일:', paths)
